# SmolVLA apple_clean53 Monitor（clean-arc 53筆 · continued from apple_v2 40k · 15k）

監控背景跑的 SmolVLA 訓練。每次想看進度 → **Restart Kernel & Run All**。

## 這個 run 是什麼

**apple 回程修法實驗**（承接「閉環回程撞平台」診斷）。用 **jitter return-recovery 資料**：放籃位置每集隨機偏移 → 回程起點多樣化，測「diversity 本身是否降低 policy 削平回程、不再撞平台」。
- task: `pick up the apple and place it into the red basket`（**apple-only**，暫不管 strawberry/banana）
- Dataset **apple_jitter_v1**：96 jitter（放籃 ±0.04x/±0.03y）+ 100 固定 = **196 ep / 148,295 frames**

## 為什麼收 jitter（背景）

閉環診斷：demo 回程對平台僅 **+9.7cm** 淨空 < policy 削平 **~10cm** → 撞平台。抬高回程 motion 兩次失敗（低點在 delicate 交接區）→ 改走 data。jitter 加回程起點多樣性，賭它降低 policy 的 mean-collapse 削平。若不夠再抬 handoff_z（B）或 offline 改軌跡（C）。

## trim_lead=12（同前）

frame0 對齊 eval reset（雙臂 home + 雙爪開 0.039，已抽樣驗證）。

## 訓練配方（沿用 apple_v2=90% baseline）

| | 說明 |
|---|---|
| Dataset | **apple_clean53（53 ep / 49,352 frames / 16D / clean-arc subset · continued from apple_v2 40k）** |
| Steps / save | **40k / 10k（4 ckpt）** |
| Recipe | `--policy.type=smolvla --policy.load_vlm_weights=true`（★ 不能用 `--policy.path`，否則 state 卡 [6]）|

## 看數據時注意

- **unique_epochs ≈ 17.3**（40k×64/148k）。
- **雙臂都動**：Action stats 左右臂 span 都 >0.5、0 個 zero-variance（handover 雙臂）。
- **Task prompts 應只有 1 條**（apple-only）。
- **閉環驗證重點**：回程有沒有變高、**不撞平台**（EEF logger 錄兩臂，比對原 apple grazes ~9.7cm）。eval 30k/40k 各測。

In [39]:
from pathlib import Path
import re
import json
import matplotlib.pyplot as plt
import pandas as pd

# ── apple_clean53：clean-arc 53筆 · continued from apple_v2 40k · 15k step ──
LOG_PATH    = Path.home() / 'openarm_isaaclab/outputs/smolvla_apple_clean53_cont_train.log'
OUTPUT_DIR  = Path.home() / 'openarm_isaaclab/outputs/finetune_smolvla_apple_clean53_cont'
TOTAL_STEPS = 40000

# 舊 runs (commented out 給未來參考):
# handover_multitask_v2 (三任務乾淨300筆, 50k) — 推薦 30k(73%均衡)，banana 隨 ckpt 變差是真的
# LOG_PATH    = Path.home() / 'openarm_isaaclab/outputs/smolvla_handover_multitask_v2_train.log'
# OUTPUT_DIR  = Path.home() / 'openarm_isaaclab/outputs/finetune_smolvla_handover_multitask_v2'
# TOTAL_STEPS = 50000
#
# apple_v2 (trim_lead=12, 固定100筆, 40k) — ✅ 閉環 40k=9/10(90%)，home attractor 已解
# v5b (16D basket_task_v2, 20k) — 85% baseline ⭐

print(f'Log: {LOG_PATH}  exists={LOG_PATH.exists()}  size={LOG_PATH.stat().st_size/1024:.1f} KB' if LOG_PATH.exists() else f'❌ Log NOT FOUND: {LOG_PATH}')
print(f'Output dir: {OUTPUT_DIR}  exists={OUTPUT_DIR.exists()}')
print(f'TOTAL_STEPS: {TOTAL_STEPS:,}')

Log: /home/teddyminelab/openarm_isaaclab/outputs/smolvla_apple_clean53_cont_train.log  exists=True  size=23.4 KB
Output dir: /home/teddyminelab/openarm_isaaclab/outputs/finetune_smolvla_apple_clean53_cont  exists=False
TOTAL_STEPS: 40,000


## 0. 訓練設定 + Dataset 資訊

從 train log 頂部 config dump + dataset `meta/info.json` 撈。靜態資訊，跑一次就夠。

In [40]:
import json
import glob
from pathlib import Path

# ── 1. 從 train log 頂部撈訓練 config ─────────────────────────────────────────
def extract_from_log(log_text: str, key: str) -> str | None:
    m = re.search(rf"'{re.escape(key)}'\s*:\s*([^,\n}}]+?)(?=,\n|\n[}}\s'])", log_text)
    return m.group(1).strip().strip("'") if m else None

KEYS_TRAIN  = ['batch_size', 'steps', 'save_freq', 'log_freq', 'num_workers', 'seed']
# SmolVLA-specific policy keys（跟 PI0.5 不同）
KEYS_POLICY = ['pretrained_path', 'chunk_size', 'n_action_steps',
               'freeze_vision_encoder', 'train_expert_only', 'load_vlm_weights',
               'compile_model', 'attention_mode',
               'expert_width_multiplier', 'num_vlm_layers', 'self_attn_every_n_layers',
               'vlm_model_name', 'use_amp',
               'optimizer_lr', 'scheduler_warmup_steps', 'scheduler_decay_steps', 'scheduler_decay_lr']
KEYS_DATASET = ['repo_id', 'root']

print("=" * 65)
print("訓練設定（從 train.log 頂部 config dump 解析）")
print("=" * 65)

train_cfg_parsed = {}
if LOG_PATH.exists():
    head = '\n'.join(LOG_PATH.read_text(errors='ignore').splitlines()[:250])
    
    print("\n── Train pipeline ──")
    for k in KEYS_TRAIN:
        v = extract_from_log(head, k)
        if v:
            train_cfg_parsed[k] = v
            print(f"  {k:30s} = {v}")
    
    print("\n── SmolVLA Policy ──")
    for k in KEYS_POLICY:
        v = extract_from_log(head, k)
        if v:
            train_cfg_parsed[k] = v
            print(f"  {k:30s} = {v}")
    
    print("\n── Dataset ──")
    for k in KEYS_DATASET:
        v = extract_from_log(head, k)
        if v:
            train_cfg_parsed[k] = v
            print(f"  {k:30s} = {v}")
    
    # rename_map（handover run 為空，basket v3/v3b 才有）
    m_rn = re.search(r"'rename_map'\s*:\s*(\{[^}]+\})", head)
    if m_rn:
        print(f"\n── Rename map ──")
        for line in m_rn.group(1).strip('{}').split(','):
            line = line.strip()
            if line:
                print(f"  {line}")
else:
    print(f"❌ Log not found: {LOG_PATH}")

# ── 2. Dataset 整體統計（從 meta/info.json 撈）─────────────────────────────
print("\n" + "=" * 65)
print("Dataset 整體統計")
print("=" * 65)

PROJECT_ROOT = Path.home() / 'openarm_isaaclab'
ds_root_str = train_cfg_parsed.get('root')
if ds_root_str:
    p = Path(ds_root_str)
    DS_ROOT = p if p.is_absolute() else (PROJECT_ROOT / p)
else:
    DS_ROOT = PROJECT_ROOT / 'datasets_lerobot/openarm/handover_3fruit_2026-06-24'

print(f"  Dataset root: {DS_ROOT}  exists={DS_ROOT.exists()}")

info_path = DS_ROOT / 'meta' / 'info.json'
total_episodes = total_frames = fps = 0
action_dim = 16
if info_path.exists():
    info = json.loads(info_path.read_text())
    total_episodes = info.get('total_episodes', 0)
    total_frames = info.get('total_frames', 0)
    fps = info.get('fps', 30)
    action_dim = int(info.get('features', {}).get('action', {}).get('shape', [16])[0])
    
    duration_min = total_frames / fps / 60 if fps else 0
    avg_frames = total_frames / total_episodes if total_episodes else 0
    avg_sec = avg_frames / fps if fps else 0
    
    print(f"\n  Dataset version    : {info.get('codebase_version')}")
    print(f"  Total episodes     : {total_episodes:,} ep")
    print(f"  Total frames       : {total_frames:,}")
    dim_marker = '(16D bimanual) ⭐' if action_dim == 16 else ('(8D single-arm) — v3 (failed)' if action_dim == 8 else '')
    print(f"  Action dim         : {action_dim} {dim_marker}")
    print(f"  FPS                : {fps}")
    print(f"  Total duration     : {duration_min:.1f} 分鐘 ({duration_min/60:.1f} 小時)")
    print(f"  Avg frames per ep  : {avg_frames:.0f} ({avg_sec:.1f} 秒/ep @ {fps} FPS)")
else:
    print(f"  ⚠️ info.json not found at {info_path}")

# ── 3. Episode length 分布 ────────────────────────────────────────────────
import pyarrow.parquet as pq
ep_files = sorted(glob.glob(str(DS_ROOT / 'meta' / 'episodes' / '**' / '*.parquet'), recursive=True))
if ep_files:
    import numpy as np
    ep_df = pq.read_table(ep_files[0], columns=['episode_index', 'length', 'tasks']).to_pandas()
    lengths = ep_df['length'].values
    print(f"\n  Episode length 分布（單位：frames）:")
    print(f"    min     = {lengths.min()}  ({lengths.min()/fps:.1f} 秒)")
    print(f"    p10     = {np.percentile(lengths, 10):.0f}")
    print(f"    median  = {np.median(lengths):.0f}  ({np.median(lengths)/fps:.1f} 秒)")
    print(f"    mean    = {lengths.mean():.0f}  ({lengths.mean()/fps:.1f} 秒)")
    print(f"    p90     = {np.percentile(lengths, 90):.0f}")
    print(f"    max     = {lengths.max()}  ({lengths.max()/fps:.1f} 秒)")

# ── 4. Features spec ─────────────────────────────────────────────────────
if info_path.exists():
    print(f"\n  Features ({len(info.get('features', {}))}):")
    for k, v in info.get('features', {}).items():
        shape = v.get('shape', '?')
        dtype = v.get('dtype', '?')
        marker = '🎥' if dtype == 'video' else ('📊' if isinstance(shape, list) and len(shape) == 1 else '  ')
        print(f"    {marker} {k:38s} {str(shape):20s} {dtype}")

# ── 5. 訓練資料覆蓋率 ─────────────────────────────────────────────────────
try:
    batch = int(train_cfg_parsed.get('batch_size', 0))
    steps = int(train_cfg_parsed.get('steps', 0))
    chunk = int(train_cfg_parsed.get('chunk_size', 50))
    if batch and steps and total_frames:
        frames_per_step = batch * chunk
        total_samples = steps * frames_per_step
        epochs_view = total_samples / total_frames
        unique_epochs = (steps * batch) / total_frames
        steps_per_epoch = total_frames / batch
        print(f"\n  訓練資料覆蓋率:")
        print(f"    每 step 採樣      : batch × chunk = {batch} × {chunk} = {frames_per_step} frame-views")
        print(f"    總 frame-views    : {steps:,} × {frames_per_step} = {total_samples:,}")
        print(f"    Frame-views/frame : {epochs_view:.1f}  (model 平均「touched」每個 frame {epochs_view:.1f} 次)")
        print(f"    Unique epochs    : {unique_epochs:.2f}  (= steps × batch / total_frames)")
        print(f"    Steps per epoch  : {steps_per_epoch:.0f}  (1 epoch ≈ 此 step 數)")
        if unique_epochs > 8:
            print(f"    ⚠️ unique_epochs={unique_epochs:.1f} 偏高 → 後段 ckpt 可能 overfit。固定位置重現下偏記憶尚可，")
            print(f"       但 benchmark 要看一段範圍（20k/30k/40k/50k）+ 三任務各自跑，別只看 last")
        elif unique_epochs > 5:
            print(f"    ⚠️ unique_epochs={unique_epochs:.1f} → benchmark 多看幾個 ckpt，別只看 last")
except (ValueError, TypeError):
    pass

# ── 6. 任務字串 ─────────────────────────────────────────────────────────────
tasks_path = DS_ROOT / 'meta' / 'tasks.parquet'
if tasks_path.exists():
    tasks_df = pd.read_parquet(tasks_path)
    print(f"\n  Task prompts ({len(tasks_df)}):")
    for task_str in tasks_df.index:
        print(f"    >>> {task_str!r}")

# ── 7. Action stats 健康檢查 ───────────────────────────────────────────────
stats_path = DS_ROOT / 'meta' / 'stats.json'
if stats_path.exists():
    stats = json.loads(stats_path.read_text())
    import numpy as np
    q01 = np.array(stats['action']['q01'])
    q99 = np.array(stats['action']['q99'])
    spans = q99 - q01
    print(f"\n  Action stats 健康檢查 ({action_dim}D, SmolVLA = absolute joint):")
    if action_dim == 16:
        # handover 雙臂：左右臂都該有 motion（span 約 1~3）；任一臂 frozen 才異常。
        # （basket 是單臂任務 → 右臂凍結才正常，此處標籤已改成 handover 版）
        right_max = spans[:8].max()
        left_max  = spans[8:].max()
        print(f"    Right arm max span (idx 0-7) : {right_max:.4f}  {'(active ✓ 雙臂任務)' if right_max > 0.5 else '(frozen?! 預期右臂會動)'}")
        print(f"    Left arm  max span (idx 8-15): {left_max:.4f}  {'(active ✓)' if left_max > 0.5 else '(too narrow!)'}")
    elif action_dim == 8:
        max_span = spans.max()
        min_span = spans.min()
        is_absolute = max_span > 0.5
        print(f"    Max q99-q01 span : {max_span:.4f}  {'(absolute joint ✓)' if is_absolute else '(too narrow!)'}") 
        print(f"    Min q99-q01 span : {min_span:.4f}  {'(gripper, narrow)' if min_span < 0.1 else ''}")
    print(f"    Narrow dims (span<0.01)    : {int((spans < 0.01).sum())} / {action_dim}  {'✓ no zero-variance' if (spans < 0.01).sum() == 0 else '⚠️ 有 zero-variance dim'}")

訓練設定（從 train.log 頂部 config dump 解析）

── Train pipeline ──
  batch_size                     = 64
  steps                          = 15000
  save_freq                      = 5000
  log_freq                       = 200
  num_workers                    = 4
  seed                           = 1000

── SmolVLA Policy ──
  pretrained_path                = /home/teddyminelab/openarm_isaaclab/outputs/finetune_smolvla_apple_v2/checkpoints/040000/pretrained_model
  chunk_size                     = 50
  n_action_steps                 = 50
  freeze_vision_encoder          = True
  train_expert_only              = True
  load_vlm_weights               = True
  compile_model                  = False
  attention_mode                 = cross_attn
  expert_width_multiplier        = 0.75
  num_vlm_layers                 = 16
  self_attn_every_n_layers       = 2
  use_amp                        = False
  optimizer_lr                   = 0.0001
  scheduler_warmup_steps         = 1000
  scheduler_decay_steps

## 1. Live progress + GPU VRAM

In [41]:
# ── Live progress + GPU VRAM ─────────────────────────────────────────────
# SmolVLA 跑得快，tqdm 顯示 'step/s' 而非 PI0.5 的 's/step'
# 範例： Training: 40%|███████████▌ | 200/500 [01:17<01:55,  2.61step/s]
# 也兼容慢的 's/step' 格式
import re
import subprocess
from datetime import timedelta

TQDM_RE = re.compile(
    r'(?P<step>\d+)/(?P<total>\d+)\s*\[(?P<elapsed>[\d:]+)<(?P<eta>[\d:]+|\?|\S+?),\s*(?P<rate>[\d.]+)(?P<unit>step/s|s/step)\]'
)

def _fmt_td(s):
    parts = s.split(':')
    if len(parts) == 2:
        parts = ['00'] + parts
    if len(parts) == 3:
        h, m, sec = parts
        return f"{int(h):2d}h {int(m):02d}m {int(sec):02d}s"
    return s

def _bar(pct: float, width: int = 40) -> str:
    filled = int(max(0, min(pct, 1.0)) * width)
    return '█' * filled + '░' * (width - filled)

# ── Block A: Training progress ────────────────────────────────────────────
print("─" * 70)
print(" TRAINING PROGRESS")
print("─" * 70)
if LOG_PATH.exists():
    raw = LOG_PATH.read_text(errors='ignore')
    matches = list(TQDM_RE.finditer(raw.replace('\r', '\n')))
    if matches:
        m = matches[-1]
        step  = int(m['step'])
        total = int(m['total'])
        pct   = step / total
        rate  = float(m['rate'])
        unit  = m['unit']
        # 統一換算成 sec/step
        if unit == 's/step':
            step_s = rate
            steps_per_s = 1.0 / rate if rate > 0 else 0
        else:  # step/s
            steps_per_s = rate
            step_s = 1.0 / rate if rate > 0 else 0
        eta_sec = (total - step) * step_s

        print(f"  [{_bar(pct)}] {pct*100:5.1f}%")
        print(f"  Step    : {step:,} / {total:,}")
        print(f"  Speed   : {step_s:.3f}s/step  ({steps_per_s:.2f} step/s)")
        print(f"  Elapsed : {_fmt_td(m['elapsed'])}")
        print(f"  ETA     : {timedelta(seconds=int(eta_sec))}  (≈ {eta_sec/60:.1f}m, {eta_sec/3600:.1f}h)")
        import time as _t
        age = int(_t.time() - LOG_PATH.stat().st_mtime)
        freshness = "🟢 live" if age < 30 else ("🟡 stale" if age < 300 else "🔴 dead?")
        print(f"  Log age : {age}s ago  {freshness}")
    else:
        print("  ⏳ Training log exists but no tqdm progress yet (model loading).")
else:
    print(f"  ❌ Log not found: {LOG_PATH}")

# ── Block B: GPU VRAM ─────────────────────────────────────────────────────
print()
print("─" * 70)
print(" GPU VRAM")
print("─" * 70)
try:
    gpu_q = subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu,power.draw',
        '--format=csv,noheader,nounits',
    ]).decode().strip().split(', ')
    name, used_mb, total_mb, util, temp, power = gpu_q
    used_gb = float(used_mb) / 1024
    total_gb = float(total_mb) / 1024
    vram_pct = float(used_mb) / float(total_mb)

    print(f"  GPU       : {name}")
    print(f"  [{_bar(vram_pct)}] {vram_pct*100:5.1f}%")
    print(f"  VRAM      : {used_gb:.2f} / {total_gb:.2f} GB  (free: {total_gb-used_gb:.2f} GB)")
    print(f"  GPU util  : {util}%   Temp: {temp}°C   Power: {power}W")

    # Per-process breakdown
    procs = subprocess.check_output([
        'nvidia-smi',
        '--query-compute-apps=pid,process_name,used_memory',
        '--format=csv,noheader,nounits',
    ]).decode().strip()
    if procs:
        print(f"\n  Processes on GPU:")
        for line in procs.splitlines():
            pid, pname, mem_mb = [x.strip() for x in line.split(',')]
            pname_short = pname.split('/')[-1] if '/' in pname else pname
            print(f"    PID {pid:>7}  {pname_short[:40]:40s}  {float(mem_mb)/1024:.2f} GB")
    else:
        print(f"\n  (No CUDA processes — GPU idle)")
except (subprocess.CalledProcessError, FileNotFoundError) as e:
    print(f"  ⚠️ nvidia-smi not available: {e}")

──────────────────────────────────────────────────────────────────────
 TRAINING PROGRESS
──────────────────────────────────────────────────────────────────────
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   1.6%
  Step    : 246 / 15,000
  Speed   : 1.380s/step  (0.72 step/s)
  Elapsed :  0h 05m 34s
  ETA     : 5:39:20  (≈ 339.3m, 5.7h)
  Log age : 1s ago  🟢 live

──────────────────────────────────────────────────────────────────────
 GPU VRAM
──────────────────────────────────────────────────────────────────────
  GPU       : NVIDIA GeForce RTX 3090
  [█████████████████████████████░░░░░░░░░░░]  74.1%
  VRAM      : 17.78 / 24.00 GB  (free: 6.22 GB)
  GPU util  : 96%   Temp: 86°C   Power: 335.51W

  Processes on GPU:
    PID 2845956  python3.12                                17.34 GB


## 2. 解析 log → loss / grad norm / LR

In [ ]:
import re
import pandas as pd
from pathlib import Path

# lerobot-train log 範例（SmolVLA）：
#   Training: 40%|...| 200/500 [01:17<01:55, 2.61step/s]INFO ... step:200 smpl:3K ep:7 epch:0.01 loss:0.693 grdn:5.331 lr:8.5e-05 updt_s:0.380 data_s:0.007
#
# 用 tqdm 的 '200/500' 拿真實 step（lerobot 把 step:2K 印成 K 格式不可靠）
PATTERN = re.compile(
    r'(?P<step>\d+)/(?P<total>\d+)\s*\[[^\]]*\].*?'
    r'loss:\s*(?P<loss>[\d.eE+\-]+).*?'
    r'grdn:\s*(?P<grdn>[\d.eE+\-]+).*?'
    r'lr:\s*(?P<lr>[\d.eE+\-]+).*?'
    r'updt_s:\s*(?P<step_s>[\d.eE+\-]+)'
)

try:
    _ = TOTAL_STEPS
except NameError:
    TOTAL_STEPS = 20000

rows = {}
if LOG_PATH.exists():
    raw = LOG_PATH.read_text(errors='ignore')
    for line in raw.replace('\r', '\n').splitlines():
        m = PATTERN.search(line)
        if m:
            step = int(m['step'])
            rows[step] = {
                'step':   step,
                'loss':   float(m['loss']),
                'grdn':   float(m['grdn']),
                'lr':     float(m['lr']),
                'step_s': float(m['step_s']),
            }

df = pd.DataFrame(sorted(rows.values(), key=lambda r: r['step']))
print(f'Parsed {len(df)} unique-step entries')
if len(df):
    latest_step = int(df['step'].iloc[-1])
    print(f'Latest step : {latest_step:,} / {TOTAL_STEPS:,}  ({latest_step/TOTAL_STEPS*100:.1f}%)')
    print(f'Latest loss : {df["loss"].iloc[-1]:.4f}')
    print(f'Min loss    : {df["loss"].min():.4f}  @ step {int(df.loc[df["loss"].idxmin(), "step"]):,}')
    mean_step_s = df['step_s'].tail(10).mean()
    print(f'Step time   : {mean_step_s:.3f}s  (last 10 entries avg)')
    eta_h = (TOTAL_STEPS - latest_step) * mean_step_s / 3600
    print(f'ETA         : {eta_h:.2f} hours ({eta_h*60:.0f} min)')
df.tail(8)

Parsed 16 unique-step entries
Latest step : 3,200 / 40,000  (8.0%)
Latest loss : 0.0430
Min loss    : 0.0430  @ step 3,200
Step time   : 1.367s  (last 10 entries avg)
ETA         : 13.97 hours (838 min)


,step,loss,grdn,lr,step_s
8,1800,0.062,0.558,0.000097,1.366
9,2000,0.059,0.590,0.000096,1.368
10,2200,0.055,0.522,0.000095,1.369
11,2400,0.053,0.571,0.000094,1.364
12,2600,0.050,0.489,0.000093,1.368
13,2800,0.048,0.483,0.000092,1.370
14,3000,0.046,0.466,0.000091,1.371
15,3200,0.043,0.447,0.000090,1.365


In [43]:
if len(df) > 5:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    axes[0, 0].plot(df['step'], df['loss'], lw=0.8, alpha=0.6, label='raw')
    axes[0, 0].plot(df['step'], df['loss'].rolling(20, min_periods=1).mean(), lw=2, color='red', label='rolling 20')
    axes[0, 0].set(xlabel='step', ylabel='loss', title='Training Loss (handover 3-fruit)')
    axes[0, 0].set_yscale('log')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(df['step'], df['grdn'], lw=0.8, alpha=0.6)
    axes[0, 1].plot(df['step'], df['grdn'].rolling(20, min_periods=1).mean(), lw=2, color='red')
    axes[0, 1].set(xlabel='step', ylabel='grad norm', title='Gradient Norm')
    axes[0, 1].set_yscale('log')
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].plot(df['step'], df['step_s'], lw=0.8)
    axes[1, 0].set(xlabel='step', ylabel='seconds/step', title='Step Time')
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].plot(df['step'], df['lr'])
    axes[1, 1].set(xlabel='step', ylabel='learning rate', title='LR Schedule (cosine + warmup)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('Not enough data points yet, wait for training to produce more log lines.')

Not enough data points yet, wait for training to produce more log lines.


## 3. Checkpoint 狀態

本 run `save_freq=10000, steps=40000` → 預期 **4 個** checkpoint（10000 … 40000）。

**1 epoch ≈ 2,317 steps**（apple_jitter_v1：148,295 frames / batch 64）：

| ckpt | 對應 epoch | eval |
|---|---|---|
| 10000 | 4.3 | ○（早期）|
| 20000 | 8.6 | ○ |
| **30000** | **12.9** | ◎ eval |
| **40000 (last)** | **17.3** | ◎ eval |

→ 閉環驗證重點：**回程對平台淨空有沒有變高（vs 原 apple grazes ~9.7cm）**，用 EEF logger 錄兩臂比對。eval 30k/40k 各測（apple→red）。flow matching variance 大，兩個 ckpt 都看。

In [37]:
ckpt_dir = OUTPUT_DIR / 'checkpoints'
if ckpt_dir.exists():
    ckpts = sorted([p.name for p in ckpt_dir.iterdir() if p.is_dir() and p.name != 'last'])
    print(f'Saved checkpoints ({len(ckpts)}): {ckpts}')
    if ckpts:
        latest = ckpt_dir / ckpts[-1]
        size_gb = sum(p.stat().st_size for p in latest.rglob('*') if p.is_file()) / 1024**3
        print(f'Latest: {ckpts[-1]}  size={size_gb:.2f} GB')
    last_link = ckpt_dir / 'last'
    if last_link.exists():
        print(f"'last' symlink → {last_link.resolve().name if last_link.is_symlink() else 'dir'}")
else:
    print(f'No checkpoint dir yet at {ckpt_dir}')

Saved checkpoints (4): ['010000', '020000', '030000', '040000']
Latest: 040000  size=1.23 GB
'last' symlink → 040000


## 4. 系統資源（即時）

In [38]:
import subprocess
print('=== RAM ===')
print(subprocess.check_output(['free', '-h']).decode())
print('=== GPU ===')
print(subprocess.check_output([
    'nvidia-smi', 
    '--query-gpu=memory.used,memory.free,memory.total,utilization.gpu,temperature.gpu',
    '--format=csv'
]).decode())
print('=== Training process ===')
try:
    out = subprocess.check_output(['pgrep', '-af', 'lerobot-train']).decode().strip()
    print(out if out else '(no lerobot-train process)')
except subprocess.CalledProcessError:
    print('(no lerobot-train process — possibly done or not started)')

=== RAM ===
               total        used        free      shared  buff/cache   available
Mem:            31Gi        16Gi       546Mi       199Mi        14Gi        14Gi
Swap:           39Gi       2.8Gi        37Gi

=== GPU ===
memory.used [MiB], memory.free [MiB], memory.total [MiB], utilization.gpu [%], temperature.gpu
5493 MiB, 18623 MiB, 24576 MiB, 72 %, 76

=== Training process ===
(no lerobot-train process — possibly done or not started)
